# Multilingual Health Q&A — Starter Notebook

**Challenge:** Multilingual Health Question Answering in Low-Resource African Languages

**Task:** Given a health question in one of 8 language/country subsets (English variants
from Ethiopia/Ghana/Kenya/Uganda, plus Akan, Amharic, Luganda, Swahili), generate a
fluent and accurate answer in the **same language**.

**Evaluation metrics** (all three computed from one generated string per row):
- `TargetRLF1` — ROUGE-L F1
- `TargetR1F1` — ROUGE-1 F1
- `TargetLLM`  — LLM-as-a-Judge score

## How this notebook fits the project

The heavy pipeline logic (FLORES-200 language mapping, TF-IDF/RAG context retrieval,
NLLB dataset prep, training, beam-search generation, submission formatting) lives in
[`src/nllb_pipeline.py`](../src/nllb_pipeline.py) as a reusable, CLI-runnable module —
that's the single source of truth for the modeling pipeline, so it isn't duplicated here.

This notebook is the **interactive front end** around it:
1. Sets up `BASE_DIR` and confirms `data/raw/` has the three dataset files, creating
   `submissions/` and `models/checkpoints/` if missing.
2. Runs quick EDA on the released data.
3. Runs a **retrieval-only** sanity check (no GPU, no training) to validate the data
   pipeline and get a baseline ROUGE number before spending GPU time.
4. Smoke-tests `src/nllb_pipeline.py` with `--dry_run` (a handful of rows, fast).
5. Kicks off the full fine-tuning run and streams its output.
6. Reloads the best checkpoint to compute a **per-subset** ROUGE breakdown on
   validation (the CLI script only prints the aggregate).
7. Validates the generated `submissions/submission_nllb.csv` against the required
   Zindi format before you upload it.

**EDA context worth keeping in mind while reading results below:**
- Only ~18k of ~29.8k training answers are unique — many are reused verbatim (canonical
  STI/contraception explainers etc.), so the RAG retriever should find good context often.
- Subsets are unevenly sized: `Eng_Uga` has 7,624 training rows vs. `Amh_Eth`'s 1,845, and
  the test set mirrors that skew (`Eng_Uga`/`Aka_Gha`/`Eng_Gha` alone are >60% of test rows) —
  watch the per-subset breakdown, not just the aggregate score.

## 1 — Install and Import Packages

In [ ]:
# Install required packages
!pip install -q scikit-learn pandas numpy rouge-score
!pip install -q transformers sentencepiece accelerate torch datasets sentence-transformers peft

import re
import json
import subprocess
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

print('Packages installed successfully')


## 2 — Set Up `BASE_DIR` and Verify Data

Works whether Jupyter's working directory is the repo root or `notebooks/` (its own
folder) — detected by checking for `src/` next to the current directory.

In [ ]:
# Universal Self-Healing Path Setup (Prevents FileNotFoundError on deleted CWD)
import os, sys
from pathlib import Path

try:
    _cwd = Path.cwd()
except (FileNotFoundError, OSError):
    os.chdir('/home/jovyan')
    _cwd = Path.cwd()

repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'

if (_cwd / 'src').exists():
    BASE_DIR = _cwd
elif (_cwd.parent / 'src').exists():
    BASE_DIR = _cwd.parent
elif (_cwd / repo_name / 'src').exists():
    BASE_DIR = _cwd / repo_name
elif (Path('/home/jovyan') / repo_name / 'src').exists():
    BASE_DIR = Path('/home/jovyan') / repo_name
else:
    BASE_DIR = Path('/home/jovyan')

os.chdir(BASE_DIR)

for path_to_add in [str(BASE_DIR), str(BASE_DIR / 'src')]:
    if path_to_add not in sys.path:
        sys.path.insert(0, path_to_add)

DATA_DIR        = BASE_DIR / 'data' / 'raw'
SUBMISSIONS_DIR = BASE_DIR / 'submissions'
CHECKPOINTS_DIR = BASE_DIR / 'models' / 'checkpoints'
SRC_PATH        = BASE_DIR / 'src' / 'nllb_pipeline.py'

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'Training set.csv'
VAL_PATH   = DATA_DIR / 'Validation set.csv'
TEST_PATH  = DATA_DIR / 'Test set.csv'

print(f'BASE_DIR : {BASE_DIR.resolve()}')
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, SRC_PATH]:
    status = 'OK' if p.exists() else 'MISSING'
    rel_p = p.relative_to(BASE_DIR) if p.is_relative_to(BASE_DIR) else p
    print(f'  [{status}] {rel_p}')


In [ ]:
from src.nllb_pipeline import (
    SUBSET_TO_FLORES,
    SUBSET_TO_NAME,
    SubsetRAGRetriever,
    build_prompt,
    compute_rouge_metrics,
    prepare_nllb_dataset,
    generate_nllb_answers,
)

print('Imported pipeline components from src/nllb_pipeline.py')
print('Subsets:', list(SUBSET_TO_NAME.keys()))


## 3 — Load Data and Quick EDA

In [ ]:
train = pd.read_csv(TRAIN_PATH)
val   = pd.read_csv(VAL_PATH)
test  = pd.read_csv(TEST_PATH)

print(f'Train shape : {train.shape}')
print(f'Val shape   : {val.shape}')
print(f'Test shape  : {test.shape}')
display(train.head(3))


In [ ]:
dist = pd.DataFrame({
    'train_n': train['subset'].value_counts(),
    'val_n': val['subset'].value_counts(),
    'test_n': test['subset'].value_counts(),
})
dist['test_pct'] = (dist['test_n'] / dist['test_n'].sum() * 100).round(1)
display(dist.sort_values('test_n', ascending=False))

n_unique_answers = train['output'].astype(str).str.strip().nunique()
print(f'\n{n_unique_answers:,} unique answers out of {len(train):,} training rows '
      f'({n_unique_answers / len(train):.1%} unique)')


## 4 — Retrieval-Only Sanity Check (no GPU, no training)

Uses `SubsetRAGRetriever` directly as an answerer (its top-1 retrieved context) to get
a baseline ROUGE number and confirm the data pipeline is sound before spending GPU time
on fine-tuning. This is *not* how the retriever is used at generation time (there it
supplies context injected into the prompt) — it's just a fast proxy metric.

In [ ]:
print('Fitting retriever on the training set...')
retriever = SubsetRAGRetriever(train, question_col='input', answer_col='output', group_col='subset')

def retrieval_only_predictions(df):
    preds = []
    for _, row in df.iterrows():
        ctx = retriever.get_context(str(row['input']), str(row['subset']))
        preds.append(ctx if ctx else '')
    return preds

val_retr_pred = retrieval_only_predictions(val)
retr_metrics = compute_rouge_metrics(val_retr_pred, val['output'].tolist())
print(f"Retrieval-only baseline — validation ROUGE-1 F1: {retr_metrics['rouge1_f1']:.4f}, "
      f"ROUGE-L F1: {retr_metrics['rougeL_f1']:.4f}")

print('\nPer-subset ROUGE-L F1:')
for subset in sorted(val['subset'].unique()):
    mask = val['subset'] == subset
    m = compute_rouge_metrics(
        [p for p, k in zip(val_retr_pred, mask) if k],
        val.loc[mask, 'output'].tolist(),
    )
    print(f'  {subset:10s} rouge1={m["rouge1_f1"]:.4f}  rougeL={m["rougeL_f1"]:.4f}')


## 5 — Smoke-Test the Pipeline (`--dry_run`)

Runs `src/nllb_pipeline.py` end-to-end on a handful of rows (40 train / 10 val / 5 test)
to catch errors — bad paths, tokenizer/version mismatches, OOM at even this tiny scale —
before committing to the full run. Takes a couple of minutes on a GPU, longer on CPU.

In [ ]:
import subprocess
dry_run_cmd = [
    sys.executable, str(SRC_PATH),
    '--dry_run',
    '--use_rag',
    '--train_path', str(TRAIN_PATH),
    '--val_path', str(VAL_PATH),
    '--test_path', str(TEST_PATH),
    '--output_dir', str(CHECKPOINTS_DIR / 'nllb-dry-run'),
    '--submission_path', str(SUBMISSIONS_DIR / 'submission_dry_run.csv'),
]
print('Running:', ' '.join(dry_run_cmd))

result = subprocess.run(dry_run_cmd, cwd=str(BASE_DIR), capture_output=True, text=True)
print(result.stdout[-4000:])
if result.returncode != 0:
    print('--- STDERR ---')
    print(result.stderr[-4000:])
assert result.returncode == 0, 'Dry run failed — fix errors above before the full run.'
print('\nDry run succeeded.')


## 6 — Full Fine-Tuning Run

This is the real, long-running job (full ~29.8k training rows, several epochs) — expect
this to take a while even on a good GPU. On AWS, consider running the equivalent command
in a `tmux`/`screen` session instead of this cell, so it survives a dropped connection:

```bash
python src/nllb_pipeline.py \
    --model_name facebook/nllb-200-distilled-600M \
    --use_rag \
    --epochs 3 \
    --batch_size 8 \
    --train_path "data/raw/Training set.csv" \
    --val_path "data/raw/Validation set.csv" \
    --test_path "data/raw/Test set.csv" \
    --output_dir models/checkpoints/nllb-health-qa-checkpoint \
    --submission_path submissions/submission_nllb.csv
```

Running it from the notebook (streams output live, but ties up this kernel for the
duration):

In [ ]:
# Execute Full Production NLLB-200 Fine-Tuning & Hybrid RAG Pipeline
import sys, subprocess

RUN_FULL_TRAINING = True

full_run_cmd = [
    sys.executable, str(SRC_PATH),
    '--model_name', 'facebook/nllb-200-distilled-600M',
    '--use_rag',
    '--use_dense_rag',
    '--min_similarity', '0.25',
    '--epochs', '3',
    '--batch_size', '8',
    '--learning_rate', '5e-5',
    '--train_path', str(TRAIN_PATH),
    '--val_path', str(VAL_PATH),
    '--test_path', str(TEST_PATH),
    '--output_dir', str(CHECKPOINTS_DIR / 'nllb-health-qa-checkpoint'),
    '--submission_path', str(SUBMISSIONS_DIR / 'submission_nllb.csv'),
]

if RUN_FULL_TRAINING:
    print('🚀 Launching Full Production Fine-Tuning & Hybrid RAG Pipeline...')
    process = subprocess.Popen(
        full_run_cmd, cwd=str(BASE_DIR),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    if process.returncode == 0:
        print('\n🎉 Full production pipeline finished successfully!')
    else:
        print(f'\n❌ Process exited with return code {process.returncode}')
else:
    print('RUN_FULL_TRAINING is False — set RUN_FULL_TRAINING = True to launch.')


## 7 — Per-Subset Validation ROUGE (from the Best Checkpoint)

`src/nllb_pipeline.py` only prints an aggregate ROUGE score. This reloads the best
checkpoint (found via `trainer_state.json`, since `load_best_model_at_end=True` doesn't
persist a copy at the top level of `output_dir`) and regenerates validation predictions
to get the per-subset breakdown that actually matters given how skewed the test set is.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

CHECKPOINT_ROOT = CHECKPOINTS_DIR / 'nllb-health-qa-checkpoint'

def find_best_checkpoint(checkpoint_root: Path, fallback_model: str = 'facebook/nllb-200-distilled-600M'):
    if not isinstance(checkpoint_root, Path):
        checkpoint_root = Path(checkpoint_root)
    if not checkpoint_root.exists():
        print(f"[NOTE] Checkpoint directory {checkpoint_root} not found. Using base model '{fallback_model}'.")
        return Path(fallback_model)
    state_files = sorted(checkpoint_root.glob('checkpoint-*/trainer_state.json'),
                          key=lambda p: p.stat().st_mtime, reverse=True)
    if not state_files:
        if (checkpoint_root / 'config.json').exists():
            return checkpoint_root
        print(f"[NOTE] No checkpoint-*/trainer_state.json found under {checkpoint_root}. Using base model '{fallback_model}'.")
        return Path(fallback_model)
    try:
        state = json.loads(state_files[0].read_text(encoding='utf-8'))
        best = state.get('best_model_checkpoint')
        if best and Path(best).exists():
            return Path(best)
        return state_files[0].parent
    except Exception:
        return state_files[0].parent
best_ckpt = find_best_checkpoint(CHECKPOINT_ROOT)
print('Best checkpoint:', best_ckpt)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_tokenizer = AutoTokenizer.from_pretrained(best_ckpt)
eval_model = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt).to(device)

retriever_full = SubsetRAGRetriever(train, question_col='input', answer_col='output', group_col='subset')

val_preds = generate_nllb_answers(eval_model, eval_tokenizer, val, retriever=retriever_full, device=device)
val_metrics = compute_rouge_metrics(val_preds, val['output'].tolist())
print(f"\nAggregate — ROUGE-1 F1: {val_metrics['rouge1_f1']:.4f}, ROUGE-L F1: {val_metrics['rougeL_f1']:.4f}")

print('\nPer-subset:')
for subset in sorted(val['subset'].unique()):
    mask = val['subset'] == subset
    m = compute_rouge_metrics(
        [p for p, k in zip(val_preds, mask) if k],
        val.loc[mask, 'output'].tolist(),
    )
    print(f'  {subset:10s} rouge1={m["rouge1_f1"]:.4f}  rougeL={m["rougeL_f1"]:.4f}')


## 8 — Validate the Submission File

In [ ]:
SUBMISSION_PATH = SUBMISSIONS_DIR / 'submission_nllb.csv'

sub = pd.read_csv(SUBMISSION_PATH)
required_cols = ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']

assert list(sub.columns) == required_cols, f'Expected columns {required_cols}, got {list(sub.columns)}'
assert len(sub) == len(test), f'Row count mismatch: {len(sub)} vs {len(test)} test rows'
assert sub[required_cols[1:]].notna().all().all(), 'Missing values found in submission'
assert set(sub['ID']) == set(test['ID']), 'Submission IDs do not match test set IDs'
assert (sub['TargetRLF1'] == sub['TargetR1F1']).all(), 'TargetRLF1 and TargetR1F1 differ'
assert (sub['TargetRLF1'] == sub['TargetLLM']).all(), 'TargetRLF1 and TargetLLM differ'

print(f'Submission OK: {SUBMISSION_PATH}')
print(f'Shape: {sub.shape}')
display(sub.head(5))


## 9 — Next Steps

- **Try `facebook/nllb-200-1.3B`** (`--model_name`) if the AWS instance has enough VRAM
  and time budget — bigger model, same pipeline, no code changes needed.
- **`--use_peft`** enables LoRA fine-tuning (falls back gracefully if `peft` isn't
  installed) — worth trying if full fine-tuning is too slow/memory-hungry for the
  instance size, or to fit more epochs in the same budget.
- **RAG threshold** — `SubsetRAGRetriever.get_context` currently returns context above a
  fixed `0.1` cosine-similarity floor. Given how templated the training answers are (see
  EDA above), it's worth checking on validation whether a higher floor (fewer, more
  relevant context injections) helps more subsets than it hurts.
- **Amharic and Akan specifically** — Amharic answers are short (~20 words avg) and
  Akan's are long (~106 words avg); watch the per-subset table in Section 7 each run
  rather than only the aggregate, since these two behave very differently during
  training.
- Confirm whether the competition rules allow calling an external hosted LLM — that
  would specifically move the `TargetLLM` (judge) score, which this pipeline doesn't
  optimize for directly.